In [3]:
from scipy.optimize import minimize
from statsmodels.tools.numdiff import approx_fprime, approx_hess
import numpy as np

def menos_logvero(theta):
    """-2 ℓ / n, a menos de constantes; theta reúne as cargas e log(psi)."""
    L = theta[:3].reshape(3, 1)
    psi = np.exp(theta[3:])
    Sigma = L @ L.T + np.diag(psi)
    return np.linalg.slogdet(Sigma)[1] + np.trace(np.linalg.solve(Sigma, R_gastos))

In [4]:
R_gastos = np.array([
    [1.00, 0.56, 0.48],
    [0.56, 1.00, 0.42],
    [0.48, 0.42, 1.00],
])

inicio = np.concatenate([np.full(3, 0.5), np.log(np.full(3, 0.5))])
ajuste_mv = minimize(
    menos_logvero,
    inicio,
    method="Newton-CG",
    jac=lambda theta: approx_fprime(theta, menos_logvero, centered=True),
    hess=lambda theta: approx_hess(theta, menos_logvero),
)

In [5]:
cargas_mv = ajuste_mv.x[:3]
psi_mv = np.exp(ajuste_mv.x[3:])
residuos_mv = R_gastos - (np.outer(cargas_mv, cargas_mv) + np.diag(psi_mv))

print("Iterações:", ajuste_mv.nit)
print("Cargas:", cargas_mv.round(3))
print("Variâncias específicas:", psi_mv.round(3))
print("Maior resíduo em módulo:", f"{np.abs(residuos_mv).max():.1e}")

Iterações: 6
Cargas: [0.8 0.7 0.6]
Variâncias específicas: [0.36 0.51 0.64]
Maior resíduo em módulo: 1.3e-10


In [6]:
0.99*0.001/(0.99*0.001 + 0.01*0.999)

0.09016393442622951